In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q vina pandas numpy pubchempy rdkit openmm openbabel-wheel
!pip install -q git+https://github.com/openmm/pdbfixer.git

import os
WORKDIR = "/content/drive/MyDrive/ctxm15_docking"
os.makedirs(WORKDIR + "/poses", exist_ok=True)
print("CPU cores:", os.cpu_count())
print("Work folder ready:", WORKDIR)

Mounted at /content/drive
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'vina' when getting requirements to build wheel
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
CPU cores: 2
Work folder ready: /content/drive/MyDrive/ctxm15_docking


In [ ]:
!pip install rdkit
import rdkit
print("rdkit OK", rdkit.__version__)


rdkit OK 2026.09.1


In [ ]:
!apt-get install -y -qq openbabel
!obabel -V
!pip install pubchempy
import pubchempy
print("pubchempy OK", pubchempy.__version__)

Open Babel 3.1.1 -- Mar 31 2024 -- 06:39:03
pubchempy OK 1.0.5


In [ ]:
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O /usr/local/bin/vina
!chmod +x /usr/local/bin/vina
!vina --version

AutoDock Vina v1.2.5


In [ ]:
!pip install py3Dmol plip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 72.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [plip]


In [ ]:
import os, subprocess, urllib.request
import numpy as np
from pdbfixer import PDBFixer
from openmm.app import PDBFile

PDB_ID = "4HBT"
SITE_RESIDUES = [70, 73, 130, 132, 166, 170, 234, 237]

raw = f"{WORKDIR}/{PDB_ID}.pdb"
if not os.path.exists(raw):
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{PDB_ID}.pdb", raw)

fixer = PDBFixer(filename=raw)
chains = list(fixer.topology.chains())
print("Chains in file:", [c.id for c in chains])

# keep only the first chain
fixer.removeChains([i for i in range(len(chains)) if i != 0])
fixer.removeHeterogens(keepWater=False)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)

clean_pdb = f"{WORKDIR}/receptor_clean.pdb"
with open(clean_pdb, "w") as fh:
    PDBFile.writeFile(fixer.topology, fixer.positions, fh)

receptor_pdbqt = f"{WORKDIR}/receptor.pdbqt"
subprocess.run(["obabel", clean_pdb, "-xr", "-O", receptor_pdbqt], check=True)

# box centre from the catalytic residues
coords = []
for line in open(clean_pdb):
    if line.startswith("ATOM") and int(line[22:26]) in SITE_RESIDUES:
        coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
center = np.mean(coords, axis=0).tolist()

n_res = len({l[22:26] for l in open(clean_pdb) if l.startswith("ATOM")})
print("Residues kept:", n_res)
print("Site atoms found:", len(coords))
print("Box centre:", [round(x, 2) for x in center])
print("Receptor file exists:", os.path.exists(receptor_pdbqt))

Chains in file: ['A', 'A']
Residues kept: 263
Site atoms found: 152
Box centre: [-5.79, -12.4, 22.31]
Receptor file exists: True


In [ ]:
!pip install pubchempy rdkit
!apt-get install -y -qq openbabel

import os, json, time, subprocess
import numpy as np
import pandas as pd
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors

WORKDIR = "/content/drive/MyDrive/ctxm15_docking"
LIG_DIR = f"{WORKDIR}/ligands"
os.makedirs(LIG_DIR, exist_ok=True)

# box centre from the cleaned receptor already on Drive
SITE_RESIDUES = [70, 73, 130, 132, 166, 170, 234, 237]
coords = []
for line in open(f"{WORKDIR}/receptor_clean.pdb"):
    if line.startswith("ATOM") and int(line[22:26]) in SITE_RESIDUES:
        coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
center = np.mean(coords, axis=0).tolist()
json.dump(center, open(f"{WORKDIR}/box_center.json", "w"))
print("Box centre saved:", [round(x, 2) for x in center])

NATURAL_COMPOUNDS = [
    "quercetin", "kaempferol", "luteolin", "apigenin", "myricetin",
    "naringenin", "catechin", "epigallocatechin gallate", "rutin", "hesperetin",
    "baicalein", "genistein", "curcumin", "resveratrol", "gallic acid",
    "ellagic acid", "caffeic acid", "rosmarinic acid", "chlorogenic acid",
    "berberine", "piperine", "emodin", "thymol", "carvacrol", "eugenol",
    "ursolic acid", "oleanolic acid", "betulinic acid", "andrographolide",
    "sanguinarine",
]
CONTROLS = ["avibactam", "clavulanic acid", "tazobactam", "cefotaxime"]

# 1) SMILES from PubChem, cached on Drive
lib_csv = f"{WORKDIR}/library.csv"
if os.path.exists(lib_csv):
    df = pd.read_csv(lib_csv)
else:
    rows = []
    for role, names in (("test", NATURAL_COMPOUNDS), ("control", CONTROLS)):
        for name in names:
            try:
                c = pcp.get_compounds(name, "name")[0]
                smi = getattr(c, "smiles", None) or getattr(c, "isomeric_smiles", None)
                rows.append((name, role, c.cid, smi))
            except Exception as e:
                print("PubChem lookup failed:", name, e)
            time.sleep(0.3)
    df = pd.DataFrame(rows, columns=["name", "role", "cid", "smiles"])
    df.to_csv(lib_csv, index=False)
print("Compounds found on PubChem:", len(df))

# 2) 3D structure and PDBQT for each ligand
info = []
for _, r in df.iterrows():
    safe = r["name"].replace(" ", "_")
    out = f"{LIG_DIR}/{safe}.pdbqt"
    mol = Chem.AddHs(Chem.MolFromSmiles(r["smiles"]))
    if not os.path.exists(out):
        if AllChem.EmbedMolecule(mol, randomSeed=42) != 0:
            print("3D build failed:", r["name"])
            continue
        AllChem.MMFFOptimizeMolecule(mol, maxIters=2000)
        tmp = f"{LIG_DIR}/{safe}.mol"
        Chem.MolToMolFile(mol, tmp)
        subprocess.run(["obabel", tmp, "-O", out], capture_output=True)
        os.remove(tmp)
    ok = os.path.exists(out) and os.path.getsize(out) > 0
    info.append((r["name"], r["role"], round(Descriptors.MolWt(mol), 1),
                 mol.GetNumHeavyAtoms(), "ok" if ok else "FAILED"))

summary = pd.DataFrame(info, columns=["name", "role", "MW", "heavy_atoms", "status"])
print(summary.to_string(index=False))
print("\nReady for docking:", (summary.status == "ok").sum(), "of", len(summary))

Box centre saved: [-5.79, -12.4, 22.31]
Compounds found on PubChem: 34
                    name    role    MW  heavy_atoms status
               quercetin    test 302.2           22     ok
              kaempferol    test 286.2           21     ok
                luteolin    test 286.2           21     ok
                apigenin    test 270.2           20     ok
               myricetin    test 318.2           23     ok
              naringenin    test 272.3           20     ok
                catechin    test 290.3           21     ok
epigallocatechin gallate    test 458.4           33     ok
                   rutin    test 610.5           43     ok
              hesperetin    test 302.3           22     ok
               baicalein    test 270.2           20     ok
               genistein    test 270.2           20     ok
                curcumin    test 368.4           27     ok
             resveratrol    test 228.2           17     ok
             gallic acid    test 170.1      

In [ ]:
import json, subprocess
WORKDIR = "/content/drive/MyDrive/ctxm15_docking"
center = json.load(open(f"{WORKDIR}/box_center.json"))
rec = f"{WORKDIR}/receptor.pdbqt"
lig = f"{WORKDIR}/ligands/quercetin.pdbqt"

print("=== LIGAND FILE ===")
print(open(lig).read())

print("=== RECEPTOR FILE (first 8 lines, then counts) ===")
lines = open(rec).read().splitlines()
print("\n".join(lines[:8]))
print("total lines:", len(lines),
      "| ATOM lines:", sum(l.startswith("ATOM") for l in lines),
      "| ROOT/BRANCH lines:", sum(l.startswith(("ROOT", "BRANCH")) for l in lines))

print("\n=== VINA RUN ===")
r = subprocess.run(["vina", "--receptor", rec, "--ligand", lig,
    "--center_x", str(center[0]), "--center_y", str(center[1]), "--center_z", str(center[2]),
    "--size_x", "20", "--size_y", "20", "--size_z", "20",
    "--exhaustiveness", "8", "--seed", "42", "--out", "/content/test_out.pdbqt"],
    capture_output=True, text=True)
print(r.stdout[-1500:])
print(r.stderr[-800:])

=== LIGAND FILE ===
REMARK  Name = 
REMARK  6 active torsions:
REMARK  status: ('A' for Active; 'I' for Inactive)
REMARK    1  A    between atoms: C_3  and  O_22
REMARK    2  A    between atoms: C_4  and  O_21
REMARK    3  A    between atoms: C_6  and  C_7
REMARK    4  A    between atoms: C_8  and  O_20
REMARK    5  A    between atoms: C_12  and  O_19
REMARK    6  A    between atoms: C_14  and  O_18
REMARK                            x       y       z     vdW  Elec       q    Type
REMARK                         _______ _______ _______ _____ _____    ______ ____
ROOT
ATOM      1  C   UNL     1       0.357   0.291   0.138  0.00  0.00    +0.000 A 
ATOM      2  C   UNL     1      -0.083   0.976   1.210  0.00  0.00    +0.000 A 
ATOM      3  C   UNL     1      -1.541   1.026   1.466  0.00  0.00    +0.000 A 
ATOM      4  O   UNL     1      -1.984   1.639   2.435  0.00  0.00    +0.000 OA
ATOM      5  C   UNL     1      -2.418   0.323   0.528  0.00  0.00    +0.000 A 
ATOM      6  C   UNL     1  

In [ ]:
import os, json, subprocess
import numpy as np

WORKDIR = "/content/drive/MyDrive/ctxm15_docking"
raw = f"{WORKDIR}/4HBT.pdb"

# delete the bad results from the misplaced box
p = f"{WORKDIR}/results_partial.csv"
if os.path.exists(p):
    os.remove(p)

SITE = {70: "SER", 73: "LYS", 130: "SER", 132: "ASN",
        166: "GLU", 170: "ASN", 234: "LYS", 237: "SER"}

# first chain only (stop at the first TER), skip alternate locations
order, xyz = [], {}
for line in open(raw):
    if line.startswith("TER") and order:
        break
    if line.startswith("ATOM") and line[16] in (" ", "A"):
        key = line[22:27]
        if key not in xyz:
            xyz[key] = []
            order.append((key, line[17:20]))
        xyz[key].append([float(line[30:38]), float(line[38:46]), float(line[46:54])])

def pick(mapping):
    return {n: (k, nm) for n, (k, nm) in mapping.items()}

# attempt 1: use the file's own numbering
by_num = {int(k[:4]): (k, nm) for k, nm in order}
ok = all(by_num.get(n, (None, None))[1] == nm for n, nm in SITE.items())
chosen = {n: by_num[n][0] for n in SITE} if ok else None
print("First residues in file:", order[:3])
print("Original numbering matches expected residues:", ok)

# attempt 2: find the Ser-Thr-Ser-Lys motif and count from it
if not ok:
    names = [nm for _, nm in order]
    hits = [i for i in range(len(names) - 3) if names[i:i+4] == ["SER", "THR", "SER", "LYS"]]
    print("STSK motif positions:", hits)
    if hits:
        i0 = hits[0]
        cand = {n: order[i0 + n - 70] for n in SITE if i0 + n - 70 < len(order)}
        if all(cand.get(n, (None, None))[1] == nm for n, nm in SITE.items()):
            chosen = {n: cand[n][0] for n in SITE}
            ok = True

if not ok:
    raise SystemExit("Could not match the catalytic residues. Paste this output to me.")

pts = np.vstack([xyz[k] for k in chosen.values()])
center = pts.mean(axis=0).tolist()
json.dump(center, open(f"{WORKDIR}/box_center.json", "w"))
print("Verified site residues:", {n: SITE[n] + str(n) for n in SITE})
print("New box centre:", [round(c, 2) for c in center])

# quick test dock of quercetin
out = "/content/test_out.pdbqt"
subprocess.run(["vina", "--receptor", f"{WORKDIR}/receptor.pdbqt",
    "--ligand", f"{WORKDIR}/ligands/quercetin.pdbqt",
    "--center_x", str(center[0]), "--center_y", str(center[1]), "--center_z", str(center[2]),
    "--size_x", "22", "--size_y", "22", "--size_z", "22",
    "--exhaustiveness", "8", "--seed", "42", "--out", out],
    capture_output=True, text=True)
for line in open(out):
    if line.startswith("REMARK VINA RESULT"):
        print("Quercetin test score:", line.split()[3])
        break

First residues in file: [('  27 ', 'THR'), ('  28 ', 'ALA'), ('  29 ', 'ASP')]
Original numbering matches expected residues: True
Verified site residues: {70: 'SER70', 73: 'LYS73', 130: 'SER130', 132: 'ASN132', 166: 'GLU166', 170: 'ASN170', 234: 'LYS234', 237: 'SER237'}
New box centre: [-6.1, -3.1, 15.84]
Quercetin test score: -7.582


In [ ]:
import os, json, subprocess, time
import pandas as pd
from google.colab import drive
from rdkit import Chem

# ---------- setup ----------
drive.mount('/content/drive')
WORKDIR = "/content/drive/MyDrive/ctxm15_docking"
LIG_DIR = f"{WORKDIR}/ligands"
POSE_DIR = f"{WORKDIR}/poses"
os.makedirs(POSE_DIR, exist_ok=True)

# install the Vina program if the runtime was reset
if subprocess.run("which vina", shell=True, capture_output=True).returncode != 0:
    os.system("wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O /usr/local/bin/vina")
    os.system("chmod +x /usr/local/bin/vina")
print(subprocess.run(["vina", "--version"], capture_output=True, text=True).stdout.strip())

# ---------- settings ----------
BOX = 22
EXHAUSTIVENESS = 8
N_MODES = 5
SEED = 42

center = json.load(open(f"{WORKDIR}/box_center.json"))
print("Box centre:", [round(c, 2) for c in center])
receptor = f"{WORKDIR}/receptor.pdbqt"
lib = pd.read_csv(f"{WORKDIR}/library.csv")

# ---------- resume from saved results ----------
partial = f"{WORKDIR}/results_partial.csv"
results = pd.read_csv(partial).to_dict("records") if os.path.exists(partial) else []
done = {r["name"] for r in results}

todo = [r for _, r in lib.iterrows()
        if r["name"] not in done
        and os.path.exists(f"{LIG_DIR}/{r['name'].replace(' ', '_')}.pdbqt")]
print(f"Already done: {len(done)} | To dock now: {len(todo)}\n")

# ---------- docking loop ----------
for r in todo:
    safe = r["name"].replace(" ", "_")
    out = f"{POSE_DIR}/{safe}.pdbqt"
    t0 = time.time()
    cmd = ["vina", "--receptor", receptor, "--ligand", f"{LIG_DIR}/{safe}.pdbqt",
           "--center_x", str(center[0]), "--center_y", str(center[1]), "--center_z", str(center[2]),
           "--size_x", str(BOX), "--size_y", str(BOX), "--size_z", str(BOX),
           "--exhaustiveness", str(EXHAUSTIVENESS), "--num_modes", str(N_MODES),
           "--seed", str(SEED), "--cpu", "0", "--out", out]
    try:
        subprocess.run(cmd, capture_output=True, text=True, timeout=1200)
        score = None
        for line in open(out):
            if line.startswith("REMARK VINA RESULT"):
                score = float(line.split()[3])
                break
        if score is None:
            print("no score for", r["name"])
            continue
        if score > -1:
            print(f"WARNING: {r['name']} scored {score}. Stop and check the box.")
        heavy = Chem.MolFromSmiles(r["smiles"]).GetNumHeavyAtoms()
        results.append({"name": r["name"], "role": r["role"], "vina_score": score,
                        "heavy_atoms": heavy,
                        "ligand_efficiency": round(score / heavy, 3)})
        pd.DataFrame(results).to_csv(partial, index=False)
        print(f"{r['name']:28s} {score:6.2f}   ({time.time() - t0:.0f}s)")
    except Exception as e:
        print("failed:", r["name"], e)

# ---------- final table ----------
final = pd.DataFrame(results).sort_values("vina_score")
final.to_csv(f"{WORKDIR}/docking_results.csv", index=False)
print("\nFinished:", len(final), "compounds")
print(final.to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
AutoDock Vina v1.2.5
Box centre: [-6.1, -3.1, 15.84]
Already done: 9 | To dock now: 25

hesperetin                    -7.32   (32s)
baicalein                     -7.60   (27s)
genistein                     -7.31   (25s)
curcumin                      -6.98   (100s)
resveratrol                   -6.61   (25s)
gallic acid                   -6.01   (17s)
ellagic acid                  -8.67   (24s)
caffeic acid                  -6.30   (19s)
rosmarinic acid               -7.80   (113s)
chlorogenic acid              -7.76   (94s)
berberine                     -7.28   (21s)
piperine                      -6.93   (26s)
emodin                        -7.63   (18s)
thymol                        -4.99   (11s)
carvacrol                     -5.21   (8s)
eugenol                       -5.69   (14s)
ursolic acid                  -6.39   (42s)
oleanolic acid                -6.6

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
WORKDIR = "/content/drive/MyDrive/ctxm15_docking"
LIG_DIR = f"{WORKDIR}/ligands"
POSE_DIR = f"{WORKDIR}/poses"
RE_DIR = f"{WORKDIR}/poses_exh32"
receptor = f"{WORKDIR}/receptor.pdbqt"
CATALYTIC = {"70", "73", "130", "132", "166", "170", "234", "237"}
POLAR = {"OA", "NA", "N", "O"}

screen = pd.read_csv(f"{WORKDIR}/docking_results.csv")

# ---- map renumbered receptor residues back to original 4HBT numbers ----
raw = []
for line in open(f"{WORKDIR}/4HBT.pdb"):
    if line.startswith("TER") and raw:
        break
    if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[16] in (" ", "A"):
        raw.append((line[17:20].strip() + line[22:27].strip(),
                    [float(line[30:38]), float(line[38:46]), float(line[46:54])]))
raw_xyz = np.array([c for _, c in raw])
label = {}
for line in open(f"{WORKDIR}/receptor_clean.pdb"):
    if line.startswith("ATOM") and line[12:16].strip() == "CA":
        xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
        d = np.linalg.norm(raw_xyz - xyz, axis=1)
        if d.min() < 0.2:
            label[int(line[22:26])] = raw[int(d.argmin())][0]
print("Residues mapped to original numbering:", len(label))

# ---- helpers ----
def read_atoms(path, first_model=True):
    atoms = []
    for line in open(path):
        if first_model and line.startswith("ENDMDL"):
            break
        if line.startswith(("ATOM", "HETATM")):
            t = line[77:79].strip()
            if t.startswith("H"):
                continue
            atoms.append({"xyz": [float(line[30:38]), float(line[38:46]), float(line[46:54])],
                          "type": t, "atom": line[12:16].strip(),
                          "resnum": int(line[22:26])})
    return atoms

rec = read_atoms(receptor, False)
rec_xyz = np.array([a["xyz"] for a in rec])

def natural(s):
    return int(re.sub(r"\D", "", s) or 0)

def analyse(pose):
    hb, near = set(), set()
    for la in read_atoms(pose):
        d = np.linalg.norm(rec_xyz - np.array(la["xyz"]), axis=1)
        for i in np.where(d < 4.0)[0]:
            near.add(label.get(rec[i]["resnum"], "?"))
        if la["type"] in POLAR:
            for i in np.where(d < 3.5)[0]:
                if rec[i]["type"] in POLAR:
                    hb.add(f"{label.get(rec[i]['resnum'], '?')}({rec[i]['atom']})")
    cat = sorted({r for r in near if re.sub(r"\D", "", r) in CATALYTIC}, key=natural)
    return (sorted(hb, key=natural), sorted(near, key=natural), cat)

def top_score(path):
    for line in open(path):
        if line.startswith("REMARK VINA RESULT"):
            return float(line.split()[3])
    return None

def rmsd(p1, p2):
    a = np.array([x["xyz"] for x in read_atoms(p1)])
    b = np.array([x["xyz"] for x in read_atoms(p2)])
    return float(np.sqrt(((a - b) ** 2).sum(axis=1).mean())) if a.shape == b.shape else np.nan

# ---- build the table ----
rows = []
for _, r in screen.iterrows():
    name = r["name"]; safe = name.replace(" ", "_")
    p8 = f"{POSE_DIR}/{safe}.pdbqt"
    p32 = f"{RE_DIR}/{safe}.pdbqt"
    have32 = os.path.exists(p32) and top_score(p32) is not None
    hb, near, cat = analyse(p8)
    rows.append({
        "name": name, "role": r["role"],
        "score_exh8": round(r["vina_score"], 2),
        "score_exh32": round(top_score(p32), 2) if have32 else np.nan,
        "pose_rmsd_A": round(rmsd(p8, p32), 2) if have32 else np.nan,
        "efficiency": r["ligand_efficiency"],
        "n_catalytic": len(cat),
        "catalytic_contacts": ", ".join(cat),
        "h_bonds": ", ".join(hb),
        "residues_within_4A": ", ".join(near),
    })

out = pd.DataFrame(rows).sort_values("score_exh8")
out.to_csv(f"{WORKDIR}/pose_analysis.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 70)
show = out[["name", "role", "score_exh8", "score_exh32", "pose_rmsd_A",
            "efficiency", "n_catalytic", "catalytic_contacts"]]
print(show.head(16).to_string(index=False))
print("\nFull table saved to", f"{WORKDIR}/pose_analysis.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Residues mapped to original numbering: 262
                    name    role  score_exh8  score_exh32  pose_rmsd_A  efficiency  n_catalytic                                    catalytic_contacts
            sanguinarine    test       -9.16        -9.16         0.02      -0.367            6         SER70, SER130, ASN132, GLU166, ASN170, SER237
epigallocatechin gallate    test       -8.95        -9.00         1.44      -0.271            7  SER70, LYS73, SER130, ASN132, GLU166, ASN170, SER237
            ellagic acid    test       -8.67        -8.65         0.09      -0.394            6         SER70, SER130, ASN132, ASN170, LYS234, SER237
                   rutin    test       -8.52          NaN          NaN      -0.198            7 SER70, SER130, ASN132, GLU166, ASN170, LYS234, SER237
                luteolin    test       -7.98        -7.97         0.02      -0

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/ctxm15_docking/pose_analysis.csv")
for n in ["sanguinarine", "epigallocatechin gallate", "ellagic acid", "luteolin",
          "myricetin", "emodin", "baicalein", "cefotaxime"]:
    r = df[df["name"] == n].iloc[0]
    print(f"{n:26s} {r['h_bonds']}")

sanguinarine               SER70(OG), ASN132(ND2), ASN170(ND2)
epigallocatechin gallate   SER70(OG), TYR105(OH), SER130(OG), SER130(O), THR216(O), GLY236(O), SER237(O), ASN132(OD1), ASN132(ND2), ASN170(OD1), ASN170(ND2), THR235(OG1), ARG274(NH1)
ellagic acid               SER70(OG), TYR105(OH), SER130(OG), THR235(O), SER237(OG), SER237(O), ASN104(ND2), ASN132(ND2), ASN170(OD1), ASN170(ND2), THR235(OG1)
luteolin                   SER70(OG), TYR105(OH), SER237(OG), ASN104(ND2), ASN132(ND2), ASN170(OD1), ASN170(ND2)
myricetin                  SER70(OG), TYR105(OH), THR216(O), SER237(OG), SER237(N), ASN104(ND2), ASN132(OD1), ASN132(ND2), GLU166(OE1), GLU166(OE2), ASN170(OD1), ASN170(ND2), THR235(OG1)
emodin                     SER70(OG), SER130(OG), SER237(O), ASN104(ND2), ASN170(OD1), THR235(OG1)
baicalein                  SER70(OG), SER130(OG), SER237(O), SER237(N), ASN104(ND2), ASN132(ND2), ASN170(OD1), ASN170(ND2)
cefotaxime                 SER70(OG), SER130(OG), SER130(O), SER237(OG),

In [ ]:
import os, subprocess
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

WORKDIR = "/content/drive/MyDrive/ctxm15_docking"
POSE_DIR = f"{WORKDIR}/poses"
CX_DIR = f"{WORKDIR}/complexes"
os.makedirs(CX_DIR, exist_ok=True)
PICKS = ["ellagic acid", "baicalein", "myricetin", "sanguinarine", "cefotaxime"]

# map prepared-file residue numbers to original 4HBT numbers, matching by CA coordinates
raw = []
for line in open(f"{WORKDIR}/4HBT.pdb"):
    if line.startswith("TER") and raw:
        break
    if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[16] in (" ", "A"):
        raw.append((int(line[22:26]), [float(line[30:38]), float(line[38:46]), float(line[46:54])]))
raw_xyz = np.array([c for _, c in raw])

new_num = {}
for line in open(f"{WORKDIR}/receptor_clean.pdb"):
    if line.startswith("ATOM") and line[12:16].strip() == "CA":
        xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
        d = np.linalg.norm(raw_xyz - xyz, axis=1)
        if d.min() < 0.2:
            new_num[int(line[22:26])] = raw[int(d.argmin())][0]
spare = max(n for n, _ in raw) + 1   # unique number for any residue that could not be matched

rec_lines = []
for line in open(f"{WORKDIR}/receptor_clean.pdb"):
    if line.startswith("ATOM"):
        n = int(line[22:26])
        if n not in new_num:
            new_num[n] = spare
            spare += 1
        rec_lines.append(f"{line[:22]}{new_num[n]:4d}{line[26:].rstrip()}")
print("Residues renumbered to original 4HBT numbering:", len(new_num))

def pose_to_hetatm(name):
    safe = name.replace(" ", "_")
    tmp = f"/content/{safe}_pose.pdb"
    subprocess.run(["obabel", f"{POSE_DIR}/{safe}.pdbqt", "-O", tmp, "-f", "1", "-l", "1"],
                   capture_output=True)
    out, k = [], 20000
    for line in open(tmp):
        if line.startswith(("ATOM", "HETATM")):
            out.append("HETATM" + f"{k:5d}" + line[11:17] + "LIG" + " Z" + "   1" + line[26:].rstrip())
            k += 1
    return out

for name in PICKS:
    lig = pose_to_hetatm(name)
    path = f"{CX_DIR}/{name.replace(' ', '_')}_complex.pdb"
    with open(path, "w") as fh:
        fh.write("\n".join(rec_lines + ["TER"] + lig + ["END"]) + "\n")
    print(f"{name:20s} ligand atoms: {len(lig):3d}  ->  {path}")

# ---- 3D viewer: grey cartoon, catalytic residues in cyan, ligand in orange ----
import py3Dmol
CAT = [70, 73, 130, 132, 166, 170, 234, 237]

def show(name):
    cx = f"{CX_DIR}/{name.replace(' ', '_')}_complex.pdb"
    v = py3Dmol.view(width=800, height=600)
    v.addModel(open(cx).read(), "pdb")
    v.setStyle({"chain": "A"}, {"cartoon": {"color": "lightgray", "opacity": 0.6}})
    v.addStyle({"chain": "A", "resi": CAT}, {"stick": {"colorscheme": "cyanCarbon"}})
    v.addStyle({"resn": "LIG"}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.2}})
    v.addResLabels({"chain": "A", "resi": CAT},
                   {"fontSize": 12, "fontColor": "black", "backgroundOpacity": 0.4})
    v.zoomTo({"resn": "LIG"})
    print("\n", name)
    v.show()

for n in PICKS:
    show(n)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Residues renumbered to original 4HBT numbering: 263
ellagic acid         ligand atoms:  26  ->  /content/drive/MyDrive/ctxm15_docking/complexes/ellagic_acid_complex.pdb
baicalein            ligand atoms:  23  ->  /content/drive/MyDrive/ctxm15_docking/complexes/baicalein_complex.pdb
myricetin            ligand atoms:  29  ->  /content/drive/MyDrive/ctxm15_docking/complexes/myricetin_complex.pdb
sanguinarine         ligand atoms:  25  ->  /content/drive/MyDrive/ctxm15_docking/complexes/sanguinarine_complex.pdb
cefotaxime           ligand atoms:  34  ->  /content/drive/MyDrive/ctxm15_docking/complexes/cefotaxime_complex.pdb

 ellagic acid


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


 baicalein


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


 myricetin


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


 sanguinarine


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


 cefotaxime


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
import os
import pandas as pd
from plip.structure.preparation import PDBComplex

WORKDIR = "/content/drive/MyDrive/ctxm15_docking"
CX_DIR = f"{WORKDIR}/complexes"
PICKS = ["ellagic acid", "baicalein", "myricetin", "sanguinarine", "cefotaxime"]

def plip_rows(name):
    cx = f"{CX_DIR}/{name.replace(' ', '_')}_complex.pdb"
    mol = PDBComplex()
    mol.load_pdb(cx)
    mol.analyze()
    rows = []
    for lig in mol.ligands:
        if lig.hetid != "LIG":
            continue
        s = mol.interaction_sets[":".join([lig.hetid, lig.chain, str(lig.position)])]

        def add(kind, items):
            for x in items:
                d = getattr(x, "distance_ad", None) or getattr(x, "distance", None)
                rows.append({"compound": name, "type": kind,
                             "residue": f"{x.restype}{x.resnr}",
                             "distance_A": round(d, 2) if d else None})

        add("H-bond (ligand donor)", s.hbonds_ldon)
        add("H-bond (protein donor)", s.hbonds_pdon)
        add("Hydrophobic", s.hydrophobic_contacts)
        add("Pi-stacking", s.pistacking)
        add("Salt bridge", list(s.saltbridge_lneg) + list(s.saltbridge_pneg))
        add("Pi-cation", list(s.pication_laro) + list(s.pication_paro))
    return rows

allrows = []
for n in PICKS:
    try:
        allrows += plip_rows(n)
    except Exception as e:
        print("PLIP failed for", n, "|", e)

tab = pd.DataFrame(allrows)
tab.to_csv(f"{WORKDIR}/plip_interactions.csv", index=False)

for n, g in tab.groupby("compound", sort=False):
    print("\n==", n, "==")
    print(g[["type", "residue", "distance_A"]].sort_values(["type", "residue"]).to_string(index=False))


== ellagic acid ==
                  type residue  distance_A
 H-bond (ligand donor)   SER70        2.99
H-bond (protein donor)  ASN104        3.32
H-bond (protein donor)  ASN132        3.36
H-bond (protein donor)  ASN170        3.67
H-bond (protein donor)  SER130        2.90
H-bond (protein donor)   SER70        4.07
H-bond (protein donor)   SER70        3.97
           Pi-stacking  TYR105        3.86
           Salt bridge  LYS234        4.32

== baicalein ==
                  type residue  distance_A
 H-bond (ligand donor)   SER70        2.94
H-bond (protein donor)  ASN104        3.59
H-bond (protein donor)  ASN132        3.27
H-bond (protein donor)  SER130        3.47
H-bond (protein donor)  SER237        4.01
H-bond (protein donor)   SER70        4.01
           Hydrophobic  THR216        3.75
           Hydrophobic  TYR105        3.63
           Hydrophobic  TYR105        3.91

== myricetin ==
                  type residue  distance_A
 H-bond (ligand donor)  ASN104        3.71


In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/complexes", "zip",
                    "/content/drive/MyDrive/ctxm15_docking/complexes")
files.download("/content/complexes.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>